In [ ]:
#! NEED TO DO

import pandas as pd
import os
import torch
import re
from eval import score_heatmap

# 1. Carregar o CSV
caminho_csv_boxes = r'C:\Repositories\picard-anomalydetection\data\BCS-DBT-boxes-validation-v2-PHASE-2-Jan-2024.csv'
df_boxes = pd.read_csv(caminho_csv_boxes)

caminho_heatmap = r'C:\Repositories\picard-anomalydetection\heatmaps\Breast-Cancer-Screening-DBT\dropout\03-04-2026_22-04-44\data\rmlo_slice000_MCD_image_3_0.5_128_256_32.pt'
nome_ficheiro = os.path.basename(caminho_heatmap)

# --- NOVA EXTRAÇÃO INTELIGENTE ---
# Procura pelo padrão "DBT-P" seguido de números
match_paciente = re.search(r'(DBT-P\d+)', nome_ficheiro)
patient_id = match_paciente.group(1) if match_paciente else None

# Procura pelas visões mamográficas padrão
match_visao = re.search(r'(rcc|lcc|rmlo|lmlo)', nome_ficheiro.lower())
view = match_visao.group(1) if match_visao else None

# Procura por "fatia" ou "slice" seguido de números em qualquer lugar do nome
match_fatia = re.search(r'(?:fatia|slice)(\d+)', nome_ficheiro.lower())

if match_fatia:
    fatia_numero = int(match_fatia.group(1))
    print(f"A procurar: Paciente {patient_id} | Visão {view} | Fatia {fatia_numero}")
    
    # 4. Procurar no CSV
    linha_tumor = df_boxes[
        (df_boxes['PatientID'] == patient_id) & 
        (df_boxes['View'].str.lower() == view) & 
        (df_boxes['Slice'] == fatia_numero) 
    ]
    
    if not linha_tumor.empty:
        print("Tumor encontrado no CSV para esta fatia!")
        y = int(linha_tumor['Y'].values[0])
        x = int(linha_tumor['X'].values[0])
        altura = int(linha_tumor['Height'].values[0])
        largura = int(linha_tumor['Width'].values[0])
        
        caixas_reais_cancro = [(y, x, altura, largura)]
        heatmap_tensor = torch.load(caminho_heatmap)
        
        mascara_perfeita, pontuacao_auc = score_heatmap(
            score_type='pixel_AUC', 
            heatmap=heatmap_tensor, 
            bboxes=caixas_reais_cancro
        )
        print(f"Pontuação AUROC: {pontuacao_auc:.4f}")
    else:
        print("Esta fatia específica não tem nenhum tumor registado no CSV.")

else:
    print(f"AVISO: O arquivo '{nome_ficheiro}' não contém o número da fatia!")
    print("Provavelmente foi gerado com o código antigo (que usava apenas a fatia do meio).")

A procurar: Paciente None | Visão rmlo | Fatia 0
Esta fatia específica não tem nenhum tumor registado no CSV.
